# Layer Normalization

This notebook works through Layer Normalization applied to a sum of vectors


## Setup functions and test data

In [1]:
using Symbolics
using LinearAlgebra
using Test
include("../src/LayerNormalization.jl")

N=8
γ = Test.Random.randn(Float32, N)
β = Test.Random.randn(Float32, N)

epsilon = 1e-6


1.0e-6

## Definitions

From ReExaminingLayerNorm:

$$\mathrm{LayerNorm}(\gamma, \beta, \epsilon, x) = \frac{x - \mu(x)}{\sqrt{\sigma^2(x) + \epsilon}} \odot \gamma + \beta$$

where $\odot$ represents the Hadamard (or elementwise) product ($\R^n \to \R^n$) $\mu$ is the mean ($\R^n \to \R$), $\sigma^2$ is the variance ($\R^n \to \R$).

By defining:
$$ c(x) = x - \mu(x)$$
$$ u_\epsilon(x) = \frac{x}{\sqrt{||x||^2 + \epsilon}}$$
$$ a_{\gamma,\beta}(x) = x \odot \gamma + \beta$$

we can write

$$\mathrm{LayerNorm}(\gamma,\beta,\epsilon, x)=a_{\gamma\sqrt{n},\beta}\circ u_{n\epsilon}\circ c(x),$$

## Centering

Here we show that $c(a+b) = c(a) + (b)$


The standard definition of $mean(v) = \mu(v)$ is:

$$\mu(v) = \frac{sum(v)}{N}$$
where $sum(v)$ is the sum over components of v, and $N$ is number of components of $v$, which results in a scalar value.

This can be stated using the dot product with the vector $\vec{1} = \{ 1,1,1,...1 \}$: $$\mu(v) = \frac{<\vec{1},v>}{N}$$

The centering operation deducts $\mu(v)$ from each component of $v$:
$$c(v) = v - μ(v) \vec{1}$$
$$ = v - \frac{<\vec{1},v>}{N} \vec{1}$$ 
$$ c(a+b)  = a + b - \frac{<\vec{1},a> + <\vec{1},b>}{N} \vec{1} = c(a) + c(b)$$

\begin{align}
<x , c(a)> &= <x, a> - \frac{<\vec{1},a><x, \vec{1}>}{N} \\
c(x) &= x - \frac{<\vec{1}, x>}{N} \vec{1} \\
\textrm{So,} \\
<c(x) , a> &= <x, a> - \frac{<\vec{1}, x><\vec{1}, a>}{N} \\
  &= <x , c(a)>
\end{align}

Test:

$$c(a + b) = c(a) + c(b)$$


In [9]:
expand(center(a+b))

(broadcast(-, broadcast(+, a, b), Ref((1//3)*Symbolics._mapreduce(identity, +, broadcast(+, a, b), Colon(), (:init => false,)))))[1:3]

In [13]:
a= Test.Random.randn(Float32, N)
b= Test.Random.randn(Float32, N)

# Check rounding error
@test center(a+b) ≈ (center(a) + center(b))

Test Passed

In [14]:
center(a+b) - (center(a) + center(b))


8-element Vector{Float32}:
 0.0
 1.1920929f-7
 0.0
 1.1920929f-7
 0.0
 1.1920929f-7
 0.0
 5.9604645f-8

## Unit Projection with Slack

$$ u_\epsilon(x) = \frac{x}{\sqrt{||x||^2 + \epsilon}}$$

This is does not generally satisfy  $ u_\epsilon(a + b) \ne u_\epsilon(a) + u_\epsilon(b)$. In fact, since $||u_\epsilon(v)||^2 \approx 1 \forall v $, if $a$ and $b$ are in the same direction, $$u_\epsilon(a)+u_\epsilon(b) \approx 2 u_\epsilon(a+b)$$

More generally, $u_\epsilon(a + b)$ is a vector in the direction of $a+b$ scaled by a factor $\lambda = \frac{1}{\sqrt{||a+b||^2 + \epsilon}}$ which depends on a and b.  $$u_\epsilon(a + b) = \lambda a + \lambda b$$ 

In [15]:
ϵ = 1e-6
u(ϵ, x) = 1 / sqrt((x ⋅ x) + ϵ) .* x
λ = 1 / sqrt((a+b) ⋅ (a+b) + ϵ)

@test u(ϵ, a+b) ≈ λ .* a + λ .* b atol=1e-7

Test Passed

In [16]:
u(ϵ, a+b) - (λ .* a + λ .* b)

8-element Vector{Float64}:
  9.53084361432488e-9
  0.0
  0.0
  0.0
  0.0
 -1.9061687117627457e-8
  4.765421779406864e-9
 -5.551115123125783e-17

## Affine Transformation

Affine transformations behave linearly up to a translation $\beta$.

\begin{align}
 a_{\gamma,\beta}(x) &= x \odot \gamma + \beta \\
 a_{\gamma,\beta}(a+b) &= (a + b) \odot \gamma  + \beta \\
 &= a \odot \gamma  + b \odot \gamma + \beta \\
 &= a_{\gamma,\beta}(a) + a_{\gamma,\beta}(b) - \beta
 \end{align}

In [20]:
affine(γ, β, a+b ) - (affine(γ, β, a )+  affine(γ, β, b ) - β)

8-element Vector{Float32}:
  0.0
 -5.9604645f-8
  0.0
  2.9802322f-8
  0.0
 -2.3841858f-7
  1.1920929f-7
  1.4901161f-8

## Layer Normalisation

\begin{align*}
\mathrm{LayerNorm}_{\gamma,\beta,\epsilon}(a+b) &= a_{\gamma\sqrt{n},\beta} \circ u_{n\epsilon} \circ c(a+b) \\
&= a_{\gamma\sqrt{n},\beta} \circ u_{n\epsilon}(c(a) + c(b)) \\
&= a_{\gamma\sqrt{n},\beta} ( \lambda c(a) + \lambda c(b) ) \\
&= a_{\gamma\sqrt{n},\beta} (\lambda c(a)) + a_{\gamma\sqrt{n},\beta}(\lambda c(b)) - \beta \\
&= \lambda c(a) \odot \gamma + \lambda c(b) \odot \gamma + \beta
\end{align*}

where $\lambda = \frac{\sqrt{N}}{\sqrt{||c(a+b)||^2 + \epsilon}}$


$$ <x , LN(a+b)> = <x, \lambda c(a) \odot \gamma> + <x, \lambda c(b) \odot \gamma> + <x, \beta>$$